# Grover's Search Algorithm — Benchmarking on IBM Quantum Hardware

> Systematic benchmarking of Grover's algorithm across ideal simulation, noisy simulation, and real IBM Quantum hardware.

This notebook imports circuit definitions and metrics from the `src/` package.

In [ ]:
# Install / upgrade dependencies (run once)
!pip3 install -q 'qiskit>=2.1.0' 'qiskit-ibm-runtime>=0.40.1' 'qiskit-aer>=0.17.0' \
    'numpy' 'pandas' 'matplotlib' 'pylatexenc' 'jinja2'

In [ ]:
# ─── Imports ───
import os, sys, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Ensure src/ is importable (handles running from src/ or project root)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# Also add CWD for when notebook runs from project root
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.circuits.grover import (
    build_oracle, build_diffusion, build_grover_circuit,
    optimal_num_iterations, theoretical_success_prob,
)
from src.metrics.compute import compute_grover_metrics, tvd
from src.simulation import (
    run_ideal, run_noisy, transpile_for_backend, run_hardware,
    SEED_TRANSPILER, SEED_SIMULATOR,
)

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService

warnings.filterwarnings('ignore', category=DeprecationWarning)
print('All imports successful.')


## Configuration

Set experiment parameters here. All sweeps, paths, and toggles are controlled from this single cell.

In [ ]:
# ===========================================
# Experiment Configuration (Profile Driven)
# ===========================================

EXPERIMENT_PROFILE = "AGGRESSIVE_FULL"  # QUICK_7MIN | STANDARD_48 | AGGRESSIVE_FULL
BACKEND_NAME = "ibm_marrakesh"
RUN_HARDWARE = True
OUTPUT_DIR = "./results/grover"

PROFILE_CONFIG = {
    "QUICK_7MIN": {
        "qubit_sweep": [2, 3, 4],
        "opt_levels": [2],
        "resilience_levels": [0],
        "dd_sweep": [False],
        "shots_sweep": [512, 1024],
        "repetitions": 1,
        "ideal_shots": 100_000,
        "max_hardware_jobs": 4,
        "hardware_budget_minutes": 7,
        "checkpoint_every": 2,
        "marked_mode": "single",
        "backend_variability": False,
        "max_backends": 1,
        "enable_zne_subset": True,
        "zne_target_limit": 2,
        "secondary_phase_enabled": False,
        "secondary_min_remaining_minutes": 30,
        "secondary_min_remaining_jobs": 1,
        "secondary_qubit_sweep": [3, 4],
        "secondary_shots_sweep": [1024],
        "secondary_repetitions": 1,
        "secondary_opt_levels": [2],
        "secondary_resilience_levels": [0],
        "secondary_dd_sweep": [False],
        "secondary_marked_states_per_qubit": 1,
    },
    "STANDARD_48": {
        "qubit_sweep": [2, 3, 4],
        "opt_levels": [0, 1, 2, 3],
        "resilience_levels": [0, 1],
        "dd_sweep": [False, True],
        "shots_sweep": [1024],
        "repetitions": 1,
        "ideal_shots": 100_000,
        "max_hardware_jobs": 48,
        "hardware_budget_minutes": 120,
        "checkpoint_every": 8,
        "marked_mode": "single",
        "backend_variability": False,
        "max_backends": 1,
        "enable_zne_subset": False,
        "zne_target_limit": 0,
        "secondary_phase_enabled": False,
        "secondary_min_remaining_minutes": 45,
        "secondary_min_remaining_jobs": 6,
        "secondary_qubit_sweep": [3, 4],
        "secondary_shots_sweep": [1024],
        "secondary_repetitions": 1,
        "secondary_opt_levels": [2, 3],
        "secondary_resilience_levels": [0, 1],
        "secondary_dd_sweep": [False, True],
        "secondary_marked_states_per_qubit": 1,
    },
    "AGGRESSIVE_FULL": {
        "qubit_sweep": [2, 3, 4, 5, 6],
        "opt_levels": [0, 1, 2, 3],
        "resilience_levels": [0, 1],
        "dd_sweep": [False, True],
        "shots_sweep": [512, 1024, 2048, 4096],
        "repetitions": 3,
        "ideal_shots": 100_000,
        "max_hardware_jobs": 400,
        "hardware_budget_minutes": 360,
        "checkpoint_every": 12,
        "marked_mode": "balanced",
        "backend_variability": True,
        "max_backends": 2,
        "enable_zne_subset": True,
        "zne_target_limit": 8,
        "secondary_phase_enabled": True,
        "secondary_min_remaining_minutes": 60,
        "secondary_min_remaining_jobs": 80,
        "secondary_qubit_sweep": [4, 5, 6],
        "secondary_shots_sweep": [1024, 2048],
        "secondary_repetitions": 1,
        "secondary_opt_levels": [1, 3],
        "secondary_resilience_levels": [0, 1],
        "secondary_dd_sweep": [False, True],
        "secondary_marked_states_per_qubit": 1,
    },
}

if EXPERIMENT_PROFILE not in PROFILE_CONFIG:
    raise ValueError(f"Unknown profile: {EXPERIMENT_PROFILE}")

CFG = PROFILE_CONFIG[EXPERIMENT_PROFILE]
QUBIT_SWEEP = CFG["qubit_sweep"]
OPT_LEVEL_SWEEP = CFG["opt_levels"]
RESILIENCE_LEVEL_SWEEP = CFG["resilience_levels"]
DD_SWEEP = CFG["dd_sweep"]
SHOTS_SWEEP = CFG["shots_sweep"]
REPETITIONS = CFG["repetitions"]
IDEAL_SHOTS = CFG["ideal_shots"]
MAX_HARDWARE_JOBS = CFG["max_hardware_jobs"]
HARDWARE_BUDGET_MINUTES = CFG["hardware_budget_minutes"]
CHECKPOINT_EVERY = CFG["checkpoint_every"]
BACKEND_VARIABILITY = CFG["backend_variability"]
MAX_BACKENDS = CFG["max_backends"]
ENABLE_ZNE_SUBSET = CFG["enable_zne_subset"]
ZNE_TARGET_CONFIG_LIMIT = CFG["zne_target_limit"]

# Secondary backend phase controls (primary backend always runs full matrix first).
SECONDARY_PHASE_ENABLED = CFG.get("secondary_phase_enabled", False)
SECONDARY_MIN_REMAINING_MINUTES = CFG.get("secondary_min_remaining_minutes", 60)
SECONDARY_MIN_REMAINING_JOBS = CFG.get(
    "secondary_min_remaining_jobs", max(1, int(0.2 * MAX_HARDWARE_JOBS))
)
SECONDARY_QUBIT_SWEEP = CFG.get("secondary_qubit_sweep", QUBIT_SWEEP[-2:])
SECONDARY_SHOTS_SWEEP = CFG.get("secondary_shots_sweep", [SHOTS_SWEEP[0], SHOTS_SWEEP[-1]])
SECONDARY_REPETITIONS = CFG.get("secondary_repetitions", 1)
SECONDARY_OPT_LEVEL_SWEEP = CFG.get("secondary_opt_levels", OPT_LEVEL_SWEEP[-2:])
SECONDARY_RESILIENCE_LEVEL_SWEEP = CFG.get("secondary_resilience_levels", RESILIENCE_LEVEL_SWEEP)
SECONDARY_DD_SWEEP = CFG.get("secondary_dd_sweep", [False, True])
SECONDARY_MARKED_STATES_PER_QUBIT = CFG.get("secondary_marked_states_per_qubit", 1)


def dedupe_keep_order(items):
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


SECONDARY_QUBIT_SWEEP = [q for q in dedupe_keep_order(SECONDARY_QUBIT_SWEEP) if q in QUBIT_SWEEP]
if not SECONDARY_QUBIT_SWEEP:
    SECONDARY_QUBIT_SWEEP = [QUBIT_SWEEP[-1]]

SECONDARY_SHOTS_SWEEP = [s for s in dedupe_keep_order(SECONDARY_SHOTS_SWEEP) if s in SHOTS_SWEEP]
if not SECONDARY_SHOTS_SWEEP:
    SECONDARY_SHOTS_SWEEP = [SHOTS_SWEEP[0]]

SECONDARY_OPT_LEVEL_SWEEP = [o for o in dedupe_keep_order(SECONDARY_OPT_LEVEL_SWEEP) if o in OPT_LEVEL_SWEEP]
if not SECONDARY_OPT_LEVEL_SWEEP:
    SECONDARY_OPT_LEVEL_SWEEP = [OPT_LEVEL_SWEEP[-1]]

SECONDARY_RESILIENCE_LEVEL_SWEEP = [
    r for r in dedupe_keep_order(SECONDARY_RESILIENCE_LEVEL_SWEEP) if r in RESILIENCE_LEVEL_SWEEP
]
if not SECONDARY_RESILIENCE_LEVEL_SWEEP:
    SECONDARY_RESILIENCE_LEVEL_SWEEP = [RESILIENCE_LEVEL_SWEEP[0]]

SECONDARY_DD_SWEEP = [d for d in dedupe_keep_order(SECONDARY_DD_SWEEP) if d in DD_SWEEP]
if not SECONDARY_DD_SWEEP:
    SECONDARY_DD_SWEEP = [DD_SWEEP[0]]

SECONDARY_REPETITIONS = max(1, int(SECONDARY_REPETITIONS))
SECONDARY_MARKED_STATES_PER_QUBIT = max(1, int(SECONDARY_MARKED_STATES_PER_QUBIT))

# Marked-state axis
MARKED_STATE_MODE = CFG["marked_mode"]  # single | balanced | custom
CUSTOM_MARKED_STATES = {
    # Example:
    # 5: [3, 17, 29],
}
DEFAULT_MARKED_STATES = {
    2: [3],
    3: [5],
    4: [10],
    5: [17],
    6: [42],
}

# For a lightweight ZNE proxy, we scale the generic noisy model and extrapolate.
ZNE_SCALE_FACTORS = [1.0, 1.5, 2.0]

# Keep hardware on by default, but allow immediate simulation-only iteration.
RUN_SIMULATION_TIERS = True


def build_marked_states_map(qubit_sweep, mode="single", custom_map=None):
    marked_map = {}
    custom_map = custom_map or {}

    for nq in qubit_sweep:
        base = DEFAULT_MARKED_STATES.get(nq, [2 ** nq - 1])
        if mode == "single":
            marked_map[nq] = [base[0]]
        elif mode == "balanced":
            # Balanced set: low weight, mixed, and high weight patterns.
            candidates = [
                1,
                base[0],
                2 ** nq - 2,
            ]
            dedup = []
            for v in candidates:
                if 0 <= v < 2 ** nq and v not in dedup:
                    dedup.append(v)
            marked_map[nq] = dedup[: min(3, len(dedup))]
        elif mode == "custom":
            vals = custom_map.get(nq, base)
            marked_map[nq] = [v for v in vals if 0 <= v < 2 ** nq]
            if not marked_map[nq]:
                marked_map[nq] = [base[0]]
        else:
            raise ValueError(f"Unsupported MARKED_STATE_MODE: {mode}")

    return marked_map


MARKED_STATES_BY_QUBIT = build_marked_states_map(
    QUBIT_SWEEP,
    mode=MARKED_STATE_MODE,
    custom_map=CUSTOM_MARKED_STATES,
)

# Backward-compatible scalar map for verification/visualization cells.
MARKED_STATES = {nq: vals[0] for nq, vals in MARKED_STATES_BY_QUBIT.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Profile: {EXPERIMENT_PROFILE}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")
print(f"Qubit sweep: {QUBIT_SWEEP}")
print(f"Shots sweep: {SHOTS_SWEEP}; repetitions={REPETITIONS}")
print(f"Marked-state mode: {MARKED_STATE_MODE}")
print("Primary-first policy:")
print(f"  Secondary phase enabled: {SECONDARY_PHASE_ENABLED}")
print(f"  Secondary thresholds: minutes>={SECONDARY_MIN_REMAINING_MINUTES}, jobs>={SECONDARY_MIN_REMAINING_JOBS}")
print(f"  Secondary subset qubits={SECONDARY_QUBIT_SWEEP}, shots={SECONDARY_SHOTS_SWEEP}, reps={SECONDARY_REPETITIONS}")

## Authentication & Backend Selection

In [ ]:
# --- Authentication ---
service = None

try:
    service = QiskitRuntimeService()
    print("Loaded saved IBM Quantum account.")
except Exception:
    pass

if service is None:
    for path in [
        "apikey.json",
        os.path.join(os.path.dirname(os.path.abspath(".")), "apikey.json"),
    ]:
        if os.path.exists(path):
            try:
                with open(path) as f:
                    apidata = json.load(f)
                token = apidata.get("apikey", "")
                if token:
                    QiskitRuntimeService.save_account(
                        channel="ibm_quantum_platform", token=token, overwrite=True
                    )
                    service = QiskitRuntimeService()
                    print(f"Loaded token from {path}")
                    break
            except Exception:
                continue

if service is None:
    from getpass import getpass

    token = getpass("Enter your IBM Quantum API token: ")
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform", token=token, overwrite=True
    )
    service = QiskitRuntimeService()
    print("Token saved and account loaded.")


# --- Backend Selection ---
def pending_jobs_safe(candidate_backend):
    try:
        return int(candidate_backend.status().pending_jobs)
    except Exception:
        return 10**9


min_qubits = max(QUBIT_SWEEP)

primary_backend = None
if BACKEND_NAME:
    try:
        primary_backend = service.backend(BACKEND_NAME)
        if primary_backend.num_qubits < min_qubits:
            print(
                f"WARNING: {BACKEND_NAME} has only {primary_backend.num_qubits} qubits "
                f"(need >= {min_qubits}). Falling back to least_busy."
            )
            primary_backend = service.least_busy(
                operational=True, simulator=False, min_num_qubits=min_qubits
            )
    except Exception as exc:
        print(f"Backend {BACKEND_NAME} unavailable ({exc}), using least_busy.")
        primary_backend = service.least_busy(
            operational=True, simulator=False, min_num_qubits=min_qubits
        )
else:
    primary_backend = service.least_busy(
        operational=True, simulator=False, min_num_qubits=min_qubits
    )

backend_pool = [primary_backend]

if BACKEND_VARIABILITY and MAX_BACKENDS > 1:
    try:
        all_candidates = service.backends(operational=True, simulator=False)
        extra_candidates = [
            b
            for b in all_candidates
            if b.num_qubits >= min_qubits and b.name != primary_backend.name
        ]
        extra_candidates = sorted(extra_candidates, key=pending_jobs_safe)

        for candidate in extra_candidates:
            if len(backend_pool) >= MAX_BACKENDS:
                break
            backend_pool.append(candidate)
    except Exception as exc:
        print(f"Backend variability discovery failed: {exc}")

# Deduplicate by backend name while preserving order.
seen = set()
dedup_pool = []
for b in backend_pool:
    if b.name not in seen:
        seen.add(b.name)
        dedup_pool.append(b)
backend_pool = dedup_pool

PRIMARY_BACKEND = backend_pool[0]
BACKEND_NAMES = [b.name for b in backend_pool]

print("Backend pool:")
for idx, b in enumerate(backend_pool, start=1):
    print(f"  [{idx}] {b.name} ({b.num_qubits} qubits)")
print(f"Primary backend: {PRIMARY_BACKEND.name}")

---
## Circuit Components

The oracle, diffusion operator, and full Grover circuit are imported from `src.circuits.grover`. Below we visualise and verify them.

In [ ]:
# ─── Optimal iteration counts ───
print('Optimal Grover iterations for 1 marked item:')
print(f"{'Qubits':>6}  {'Search Space':>12}  {'Iterations':>10}  {'Success Prob':>12}")
print('-' * 46)
for n in QUBIT_SWEEP:
    k = optimal_num_iterations(n)
    p = theoretical_success_prob(n, k)
    print(f'{n:>6}  {2**n:>12}  {k:>10}  {p:>12.4f}')


In [ ]:
# ─── Circuit visualisation ───
for nq in QUBIT_SWEEP:
    qc = build_grover_circuit(nq, MARKED_STATES[nq])
    print(f'\nGrover {nq}q, marked=|{format(MARKED_STATES[nq], f"0{nq}b")}⟩, '
          f'{optimal_num_iterations(nq)} iterations:')
    display(qc.draw(output='mpl', fold=-1))


In [ ]:
# ─── Ideal simulation verification ───
for nq in QUBIT_SWEEP:
    qc = build_grover_circuit(nq, MARKED_STATES[nq])
    ideal = run_ideal(qc, shots=10_000)
    marked_bs = format(MARKED_STATES[nq], f'0{nq}b')
    print(f'{nq}q: P(|{marked_bs}⟩) = {ideal.get(marked_bs, 0.0):.4f}')


---
## Experiment Sweep

48-configuration factorial sweep: 4 opt levels x 3 qubit sizes x 2 resilience x 2 DD.

In [ ]:
# ===========================================
# Run Expanded Experiments
# ===========================================
from itertools import product
import time

from src.simulation import make_generic_noise_model

all_results = []
all_distributions = {}
run_counter = 0
hardware_jobs_submitted = 0
hardware_budget_exhausted = False
zne_evaluations = 0
experiment_start = time.time()

SECONDARY_PHASE_EXECUTED = False
SECONDARY_PHASE_REASON = "not_evaluated"
SECONDARY_PHASE_BACKENDS = []


def zne_proxy_success_estimate(circuit, ideal_dist, num_qubits, marked_state, shots):
    """Lightweight ZNE proxy on noisy simulation by scaling depolarizing noise.

    This keeps runtime predictable for notebook usage while still adding
    a mitigation comparison axis for thesis discussion.
    """
    scale_to_success = {}
    for scale in ZNE_SCALE_FACTORS:
        err_1q = min(0.49, 0.001 * scale)
        err_2q = min(0.49, 0.01 * scale)
        scaled_noise = make_generic_noise_model(err_1q=err_1q, err_2q=err_2q)
        dist_scaled = run_noisy(circuit, shots=shots, noise_model=scaled_noise)
        metrics_scaled = compute_grover_metrics(
            dist_scaled,
            ideal_dist,
            num_qubits,
            marked_state,
        )
        scale_to_success[scale] = metrics_scaled["success_prob"]

    x = np.array(sorted(scale_to_success.keys()), dtype=float)
    y = np.array([scale_to_success[k] for k in x], dtype=float)
    coeff = np.polyfit(x, y, 1)
    est_zero_noise = float(np.clip(coeff[1], 0.0, 1.0))

    return {
        "zne_scales": list(x),
        "zne_success_by_scale": {str(k): float(v) for k, v in scale_to_success.items()},
        "zne_success_estimate": est_zero_noise,
        "zne_fit_slope": float(coeff[0]),
    }


def checkpoint_results(rows, output_dir, suffix):
    if not rows:
        return
    ckpt_df = pd.DataFrame(rows)
    ckpt_csv = os.path.join(output_dir, f"grover_results_checkpoint_{suffix}.csv")
    ckpt_df.to_csv(ckpt_csv, index=False)
    print(f"Checkpoint saved: {ckpt_csv}")


def remaining_budget_state():
    elapsed_minutes = (time.time() - experiment_start) / 60.0
    remaining_minutes = max(0.0, HARDWARE_BUDGET_MINUTES - elapsed_minutes)
    remaining_jobs = max(0, MAX_HARDWARE_JOBS - hardware_jobs_submitted)
    return elapsed_minutes, remaining_minutes, remaining_jobs


def can_start_secondary_phase():
    if not SECONDARY_PHASE_ENABLED:
        return False, "secondary_phase_disabled"
    if len(backend_pool) <= 1:
        return False, "no_secondary_backend_available"
    if not RUN_HARDWARE:
        return False, "hardware_disabled"
    if hardware_budget_exhausted:
        return False, "hardware_budget_already_exhausted"

    _, remaining_minutes, remaining_jobs = remaining_budget_state()
    if remaining_minutes < SECONDARY_MIN_REMAINING_MINUTES:
        return False, f"insufficient_remaining_minutes:{remaining_minutes:.1f}"
    if remaining_jobs < SECONDARY_MIN_REMAINING_JOBS:
        return False, f"insufficient_remaining_jobs:{remaining_jobs}"

    return True, "secondary_phase_enabled"


def marked_states_for_phase(num_qubits, phase_name):
    base_states = MARKED_STATES_BY_QUBIT.get(num_qubits, [2 ** num_qubits - 1])
    if phase_name == "secondary":
        return base_states[:SECONDARY_MARKED_STATES_PER_QUBIT]
    return base_states


phase_plan = [
    {
        "name": "primary",
        "scope": "primary_full",
        "backends": [PRIMARY_BACKEND],
        "qubit_sweep": QUBIT_SWEEP,
        "shots_sweep": SHOTS_SWEEP,
        "repetitions": REPETITIONS,
        "opt_levels": OPT_LEVEL_SWEEP,
        "resilience_levels": RESILIENCE_LEVEL_SWEEP,
        "dd_sweep": DD_SWEEP,
    }
]

secondary_ok, secondary_reason = can_start_secondary_phase()
if secondary_ok:
    SECONDARY_PHASE_EXECUTED = True
    SECONDARY_PHASE_REASON = "started"
    SECONDARY_PHASE_BACKENDS = [b.name for b in backend_pool[1:]]
    phase_plan.append(
        {
            "name": "secondary",
            "scope": "secondary_subset",
            "backends": backend_pool[1:],
            "qubit_sweep": SECONDARY_QUBIT_SWEEP,
            "shots_sweep": SECONDARY_SHOTS_SWEEP,
            "repetitions": SECONDARY_REPETITIONS,
            "opt_levels": SECONDARY_OPT_LEVEL_SWEEP,
            "resilience_levels": SECONDARY_RESILIENCE_LEVEL_SWEEP,
            "dd_sweep": SECONDARY_DD_SWEEP,
        }
    )
else:
    SECONDARY_PHASE_EXECUTED = False
    SECONDARY_PHASE_REASON = secondary_reason

print("=" * 64)
print("Grover expanded sweep started")
print(f"Profile={EXPERIMENT_PROFILE} | RUN_HARDWARE={RUN_HARDWARE}")
print(f"Backend pool={BACKEND_NAMES}")
print(f"Phase plan={[p['name'] for p in phase_plan]}")
if not SECONDARY_PHASE_EXECUTED:
    print(f"Secondary phase skipped: {SECONDARY_PHASE_REASON}")
print("=" * 64)

for phase in phase_plan:
    print(f"\n[Phase: {phase['name']}] scope={phase['scope']}")

    for backend in phase["backends"]:
        print(f"\nBackend: {backend.name}")

        for nq in phase["qubit_sweep"]:
            for marked in marked_states_for_phase(nq, phase["name"]):
                num_iter = optimal_num_iterations(nq)
                marked_bs = format(marked, f"0{nq}b")
                qc = build_grover_circuit(num_qubits=nq, marked_state=marked)

                print(
                    f"\n{'-' * 64}\n"
                    f"Qubits={nq}, marked=|{marked_bs}>, iterations={num_iter}, backend={backend.name}\n"
                    f"{'-' * 64}"
                )

                ideal_dist = run_ideal(qc, shots=IDEAL_SHOTS)
                ideal_metrics = compute_grover_metrics(ideal_dist, ideal_dist, nq, marked)
                print(f"Ideal success P(|{marked_bs}>) = {ideal_metrics['success_prob']:.4f}")

                transpiled_cache = {}
                transpiled_metrics_cache = {}
                for opt_level in phase["opt_levels"]:
                    tc, t_metrics = transpile_for_backend(qc, backend, opt_level)
                    transpiled_cache[opt_level] = tc
                    transpiled_metrics_cache[opt_level] = t_metrics

                for shot_count, opt_level, res_level, dd_enabled, rep_idx in product(
                    phase["shots_sweep"],
                    phase["opt_levels"],
                    phase["resilience_levels"],
                    phase["dd_sweep"],
                    range(1, phase["repetitions"] + 1),
                ):
                    run_counter += 1
                    run_id = f"run-{run_counter:05d}"
                    timestamp = datetime.utcnow().isoformat() + "Z"
                    t_metrics = transpiled_metrics_cache[opt_level]

                    row = {
                        "run_id": run_id,
                        "timestamp": timestamp,
                        "profile": EXPERIMENT_PROFILE,
                        "algorithm": "grover",
                        "backend": backend.name,
                        "backend_phase": phase["name"],
                        "backend_scope": phase["scope"],
                        "num_qubits": nq,
                        "marked_state": marked,
                        "marked_bitstring": marked_bs,
                        "num_iterations": num_iter,
                        "search_space": 2 ** nq,
                        "opt_level": opt_level,
                        "resilience_level": res_level,
                        "dd_enable": dd_enabled,
                        "shots": shot_count,
                        "rep_idx": rep_idx,
                        "run_hardware": RUN_HARDWARE,
                        "depth_2q": t_metrics["depth_2q"],
                        "count_2q": t_metrics["count_2q"],
                        "total_depth": t_metrics["total_depth"],
                        "ideal_success_prob": ideal_metrics["success_prob"],
                        "ideal_amplification": ideal_metrics["amplification"],
                        "skip_reason": None,
                    }

                    noisy_dist = run_noisy(qc, backend=backend, shots=shot_count)
                    noisy_metrics = compute_grover_metrics(noisy_dist, ideal_dist, nq, marked)
                    row["noisy_success_prob"] = noisy_metrics["success_prob"]
                    row["tvd_noisy_vs_ideal"] = noisy_metrics["tvd_vs_ideal"]
                    row["noisy_amplification"] = noisy_metrics["amplification"]

                    do_zne_subset = (
                        ENABLE_ZNE_SUBSET
                        and phase["name"] == "primary"
                        and zne_evaluations < ZNE_TARGET_CONFIG_LIMIT
                        and rep_idx == 1
                        and shot_count == phase["shots_sweep"][0]
                        and opt_level == phase["opt_levels"][-1]
                    )

                    if do_zne_subset:
                        zne_data = zne_proxy_success_estimate(
                            qc,
                            ideal_dist,
                            nq,
                            marked,
                            shots=shot_count,
                        )
                        row["zne_mode"] = "noisy_proxy"
                        row["zne_success_estimate"] = zne_data["zne_success_estimate"]
                        row["zne_fit_slope"] = zne_data["zne_fit_slope"]
                        row["zne_scales"] = json.dumps(zne_data["zne_scales"])
                        row["zne_success_by_scale"] = json.dumps(zne_data["zne_success_by_scale"])
                        zne_evaluations += 1
                    else:
                        row["zne_mode"] = "off"
                        row["zne_success_estimate"] = None
                        row["zne_fit_slope"] = None
                        row["zne_scales"] = None
                        row["zne_success_by_scale"] = None

                    can_run_hw = RUN_HARDWARE and not hardware_budget_exhausted
                    if can_run_hw:
                        elapsed_minutes, _, _ = remaining_budget_state()
                        if hardware_jobs_submitted >= MAX_HARDWARE_JOBS:
                            hardware_budget_exhausted = True
                            can_run_hw = False
                            row["skip_reason"] = "max_hardware_jobs_reached"
                        elif elapsed_minutes >= HARDWARE_BUDGET_MINUTES:
                            hardware_budget_exhausted = True
                            can_run_hw = False
                            row["skip_reason"] = "hardware_time_budget_reached"

                    if can_run_hw:
                        tc = transpiled_cache[opt_level]
                        try:
                            hw_dist, job_id = run_hardware(
                                tc,
                                backend,
                                shots=shot_count,
                                resilience_level=res_level,
                                dd_enable=dd_enabled,
                            )
                            hardware_jobs_submitted += 1

                            hw_metrics = compute_grover_metrics(
                                hw_dist,
                                ideal_dist,
                                nq,
                                marked,
                                noisy_dist=noisy_dist,
                            )
                            row["hw_success_prob"] = hw_metrics["success_prob"]
                            row["tvd_hw_vs_ideal"] = hw_metrics["tvd_vs_ideal"]
                            row["tvd_hw_vs_noisy"] = hw_metrics.get("tvd_vs_noisy")
                            row["hw_amplification"] = hw_metrics["amplification"]
                            row["job_id"] = job_id

                            all_distributions[run_id] = {
                                "ideal": ideal_dist,
                                "noisy": noisy_dist,
                                "hardware": hw_dist,
                            }

                            print(
                                f"[{run_id}] phase={phase['name']} shot={shot_count} opt={opt_level} "
                                f"res={res_level} dd={'ON' if dd_enabled else 'OFF'} rep={rep_idx} "
                                f"success={hw_metrics['success_prob']:.4f}"
                            )
                        except Exception as exc:
                            row["hw_success_prob"] = None
                            row["tvd_hw_vs_ideal"] = None
                            row["tvd_hw_vs_noisy"] = None
                            row["hw_amplification"] = None
                            row["job_id"] = None
                            row["skip_reason"] = f"hardware_error:{type(exc).__name__}"
                            print(f"[{run_id}] Hardware execution failed: {exc}")
                    else:
                        row["hw_success_prob"] = None
                        row["tvd_hw_vs_ideal"] = None
                        row["tvd_hw_vs_noisy"] = None
                        row["hw_amplification"] = None
                        row["job_id"] = None
                        if row["skip_reason"] is None:
                            if not RUN_HARDWARE:
                                row["skip_reason"] = "hardware_disabled"
                            elif hardware_budget_exhausted:
                                row["skip_reason"] = "budget_exhausted_before_submission"
                            else:
                                row["skip_reason"] = "phase_guard_blocked"

                    all_results.append(row)

                    if run_counter % CHECKPOINT_EVERY == 0:
                        checkpoint_results(all_results, OUTPUT_DIR, suffix=f"{run_counter:05d}")

print("\n" + "=" * 64)
print(f"Total attempted rows: {run_counter}")
print(f"Hardware jobs submitted: {hardware_jobs_submitted}")
print(f"Hardware budget exhausted: {hardware_budget_exhausted}")
print(f"ZNE subset evaluations: {zne_evaluations}")
print(f"Secondary phase executed: {SECONDARY_PHASE_EXECUTED}")
print(f"Secondary phase reason: {SECONDARY_PHASE_REASON}")
print("=" * 64)

---
## Results & Analysis

In [ ]:
# --- Build DataFrame ---
df = pd.DataFrame(all_results)
print(f"Rows collected: {len(df)}")
if not df.empty:
    display(df.head(20))

# --- Run manifest ---
manifest = {
    "profile": EXPERIMENT_PROFILE,
    "run_hardware": RUN_HARDWARE,
    "rows_total": int(len(df)),
    "rows_with_hardware": int(df["hw_success_prob"].notna().sum()) if not df.empty else 0,
    "rows_skipped": int(df["skip_reason"].notna().sum()) if not df.empty else 0,
    "backends": BACKEND_NAMES,
    "primary_backend": PRIMARY_BACKEND.name,
    "backend_phase_policy": "primary_full_then_secondary_subset",
    "secondary_phase_enabled": SECONDARY_PHASE_ENABLED,
    "secondary_phase_executed": bool(globals().get("SECONDARY_PHASE_EXECUTED", False)),
    "secondary_phase_reason": globals().get("SECONDARY_PHASE_REASON", "unknown"),
    "secondary_phase_backends": globals().get("SECONDARY_PHASE_BACKENDS", []),
    "secondary_phase_thresholds": {
        "min_remaining_minutes": SECONDARY_MIN_REMAINING_MINUTES,
        "min_remaining_jobs": SECONDARY_MIN_REMAINING_JOBS,
    },
    "secondary_subset": {
        "qubit_sweep": SECONDARY_QUBIT_SWEEP,
        "shots_sweep": SECONDARY_SHOTS_SWEEP,
        "repetitions": SECONDARY_REPETITIONS,
        "opt_levels": SECONDARY_OPT_LEVEL_SWEEP,
        "resilience_levels": SECONDARY_RESILIENCE_LEVEL_SWEEP,
        "dd_sweep": SECONDARY_DD_SWEEP,
        "marked_states_per_qubit": SECONDARY_MARKED_STATES_PER_QUBIT,
    },
    "qubit_sweep": QUBIT_SWEEP,
    "shots_sweep": SHOTS_SWEEP,
    "repetitions": REPETITIONS,
    "opt_levels": OPT_LEVEL_SWEEP,
    "resilience_levels": RESILIENCE_LEVEL_SWEEP,
    "dd_sweep": DD_SWEEP,
    "marked_state_mode": MARKED_STATE_MODE,
    "zne_enabled": ENABLE_ZNE_SUBSET,
}
print("Run manifest:")
print(json.dumps(manifest, indent=2))

# --- Export CSV ---
csv_path = os.path.join(OUTPUT_DIR, "grover_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved {csv_path}")

# --- Export JSON ---
json_payload = {
    "created": datetime.utcnow().isoformat() + "Z",
    "profile": EXPERIMENT_PROFILE,
    "manifest": manifest,
    "grover_params": {
        "marked_states_by_qubit": {
            str(k): v for k, v in MARKED_STATES_BY_QUBIT.items()
        }
    },
    "runs": all_results,
    "distributions": {
        rid: {tier: dist for tier, dist in tiers.items()}
        for rid, tiers in all_distributions.items()
    },
}
json_path = os.path.join(OUTPUT_DIR, "grover_results.json")
with open(json_path, "w") as f:
    json.dump(json_payload, f, indent=2, default=str)
print(f"Saved {json_path}")

manifest_path = os.path.join(OUTPUT_DIR, "grover_run_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved {manifest_path}")

# --- Analysis frame for plotting ---
analysis_df = df.copy()
if not analysis_df.empty:
    analysis_df = analysis_df[analysis_df["backend"] == PRIMARY_BACKEND.name]
    analysis_df = analysis_df[analysis_df["rep_idx"] == 1]
    analysis_df = analysis_df[analysis_df["shots"] == max(SHOTS_SWEEP)]

print(f"Analysis rows after filtering: {len(analysis_df)}")

In [ ]:
# ===========================================
# Plots (Profile-aware, Aggregated)
# ===========================================
plot_df = analysis_df.copy() if "analysis_df" in globals() else df.copy()
if plot_df.empty:
    print("No data available for plotting.")
else:
    agg = (
        plot_df.groupby(["num_qubits", "opt_level"], as_index=False)
        .agg(
            hw_success_prob=("hw_success_prob", "mean"),
            noisy_success_prob=("noisy_success_prob", "mean"),
            ideal_success_prob=("ideal_success_prob", "mean"),
            tvd_hw_vs_ideal=("tvd_hw_vs_ideal", "mean"),
            tvd_noisy_vs_ideal=("tvd_noisy_vs_ideal", "mean"),
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Plot 1: Success Probability ---
    ax = axes[0]
    for nq in sorted(agg["num_qubits"].unique()):
        sub = agg[agg["num_qubits"] == nq]
        if sub["hw_success_prob"].notna().any():
            ax.plot(
                sub["opt_level"],
                sub["hw_success_prob"],
                marker="o",
                label=f"HW (qubits={nq})",
            )
        if sub["noisy_success_prob"].notna().any():
            ax.plot(
                sub["opt_level"],
                sub["noisy_success_prob"],
                marker="s",
                linestyle="--",
                alpha=0.6,
                label=f"Noisy (qubits={nq})",
            )
        if sub["ideal_success_prob"].notna().any():
            ax.axhline(
                sub["ideal_success_prob"].iloc[0],
                linestyle=":",
                alpha=0.3,
                label=f"Ideal (qubits={nq})",
            )

    ax.set_xlabel("Optimization Level")
    ax.set_ylabel("Success Probability")
    ax.set_title(f"Grover Success vs Opt Level ({EXPERIMENT_PROFILE})")
    ax.legend(fontsize=7, loc="best")
    ax.grid(True, alpha=0.3)

    # --- Plot 2: TVD ---
    ax = axes[1]
    for nq in sorted(agg["num_qubits"].unique()):
        sub = agg[agg["num_qubits"] == nq]
        if sub["tvd_hw_vs_ideal"].notna().any():
            ax.plot(
                sub["opt_level"],
                sub["tvd_hw_vs_ideal"],
                marker="o",
                label=f"HW vs Ideal (qubits={nq})",
            )
        if sub["tvd_noisy_vs_ideal"].notna().any():
            ax.plot(
                sub["opt_level"],
                sub["tvd_noisy_vs_ideal"],
                marker="s",
                linestyle="--",
                alpha=0.6,
                label=f"Noisy vs Ideal (qubits={nq})",
            )

    ax.set_xlabel("Optimization Level")
    ax.set_ylabel("Total Variation Distance")
    ax.set_title(f"Grover TVD vs Opt Level ({EXPERIMENT_PROFILE})")
    ax.legend(fontsize=7, loc="best")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "success_prob_and_tvd.png")
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {fig_path}")

    # --- Optional Plot: Shot Scaling Trend ---
    if df["shots"].nunique() > 1:
        shot_agg = (
            df.groupby(["num_qubits", "shots"], as_index=False)
            .agg(
                hw_success_prob=("hw_success_prob", "mean"),
                noisy_success_prob=("noisy_success_prob", "mean"),
            )
            .sort_values(["num_qubits", "shots"])
        )

        fig2, ax2 = plt.subplots(figsize=(10, 5))
        for nq in sorted(shot_agg["num_qubits"].unique()):
            sub = shot_agg[shot_agg["num_qubits"] == nq]
            if sub["hw_success_prob"].notna().any():
                ax2.plot(
                    sub["shots"],
                    sub["hw_success_prob"],
                    marker="o",
                    label=f"HW (qubits={nq})",
                )
            if sub["noisy_success_prob"].notna().any():
                ax2.plot(
                    sub["shots"],
                    sub["noisy_success_prob"],
                    marker="s",
                    linestyle="--",
                    alpha=0.6,
                    label=f"Noisy (qubits={nq})",
                )

        ax2.set_xlabel("Shots")
        ax2.set_ylabel("Mean Success Probability")
        ax2.set_title(f"Grover Shot Scaling ({EXPERIMENT_PROFILE})")
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=8, loc="best")

        plt.tight_layout()
        shot_fig_path = os.path.join(OUTPUT_DIR, "shot_scaling_success.png")
        fig2.savefig(shot_fig_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved {shot_fig_path}")

In [ ]:
# --- Bar chart: success probability by representative configuration ---
bar_df = analysis_df.copy() if "analysis_df" in globals() else df.copy()
if bar_df.empty:
    print("No data available for bar chart.")
else:
    if len(bar_df) > 30:
        bar_df = bar_df.sort_values("hw_success_prob", ascending=False).head(30)

    fig, ax = plt.subplots(figsize=(14, 5))

    labels, hw_vals, noisy_vals, ideal_vals = [], [], [], []
    for _, row in bar_df.iterrows():
        tag = (
            f"q={int(row['num_qubits'])} m={int(row['marked_state'])} "
            f"opt={int(row['opt_level'])} res={int(row['resilience_level'])} "
            f"dd={'Y' if row['dd_enable'] else 'N'}"
        )
        labels.append(tag)
        hw_vals.append(row.get("hw_success_prob") or 0.0)
        noisy_vals.append(row.get("noisy_success_prob") or 0.0)
        ideal_vals.append(row.get("ideal_success_prob") or 0.0)

    x = np.arange(len(labels))
    w = 0.25
    ax.bar(x - w, ideal_vals, w, label="Ideal", alpha=0.7)
    ax.bar(x, noisy_vals, w, label="Noisy", alpha=0.7)
    ax.bar(x + w, hw_vals, w, label="Hardware", alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=90, fontsize=6)
    ax.set_ylabel("Success Probability")
    ax.set_title(f"Grover Success by Configuration ({EXPERIMENT_PROFILE})")
    ax.legend()
    ax.grid(True, alpha=0.2, axis="y")

    plt.tight_layout()
    bar_path = os.path.join(OUTPUT_DIR, "success_prob_bar.png")
    fig.savefig(bar_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {bar_path}")

In [ ]:
# --- Summary Table ---
summary_df = analysis_df.copy() if "analysis_df" in globals() else df.copy()
summary_cols = [
    "run_id",
    "profile",
    "backend",
    "backend_phase",
    "backend_scope",
    "num_qubits",
    "num_iterations",
    "marked_state",
    "shots",
    "rep_idx",
    "opt_level",
    "resilience_level",
    "dd_enable",
    "depth_2q",
    "count_2q",
    "ideal_success_prob",
    "noisy_success_prob",
    "hw_success_prob",
    "tvd_hw_vs_ideal",
    "tvd_noisy_vs_ideal",
    "hw_amplification",
    "zne_mode",
    "zne_success_estimate",
    "skip_reason",
    "job_id",
]
existing_cols = [c for c in summary_cols if c in summary_df.columns]

if summary_df.empty:
    print("No rows available for summary table.")
else:
    try:
        display(
            summary_df[existing_cols]
            .style.format(precision=4, na_rep="-")
            .set_caption(f"Grover Expanded Sweep Summary ({EXPERIMENT_PROFILE})")
        )
    except AttributeError:
        display(summary_df[existing_cols])

---
## Search Verification & Amplification Analysis

Using the **best hardware run** (highest success probability), verify that Grover's algorithm successfully amplified the marked state above the random-guessing baseline.

In [ ]:
# --- Search Verification from Best Hardware Run ---
verification_df = analysis_df.copy() if "analysis_df" in globals() else df.copy()

if RUN_HARDWARE and not verification_df.empty and verification_df["hw_success_prob"].notna().any():
    candidate_df = verification_df[verification_df["hw_success_prob"].notna()].copy()
    best_row = candidate_df.loc[candidate_df["hw_success_prob"].idxmax()]

    best_id = best_row["run_id"]
    best_nq = int(best_row["num_qubits"])
    best_marked = int(best_row["marked_state"])
    hw_dist = all_distributions.get(best_id, {}).get("hardware", {})

    if hw_dist:
        marked_bs = format(best_marked, f"0{best_nq}b")
        random_p = 1.0 / (2 ** best_nq)
        measured_p = hw_dist.get(marked_bs, 0.0)
        amp_ratio = measured_p / random_p if random_p > 0 else 0.0

        print(
            f"Best run: {best_id} (backend={best_row['backend']}, qubits={best_nq}, "
            f"marked=|{marked_bs}>, opt={int(best_row['opt_level'])}, "
            f"res={int(best_row['resilience_level'])}, "
            f"dd={'ON' if best_row['dd_enable'] else 'OFF'}, shots={int(best_row['shots'])})"
        )
        print(f"  Random guessing baseline: {random_p:.4f}")
        print(f"  Measured P(|{marked_bs}>) : {measured_p:.4f}")
        print(f"  Amplification ratio        : {amp_ratio:.2f}x")
        print(f"  Search status              : {'SUCCESSFUL' if measured_p > random_p else 'FAILED'}")

        fig, ax = plt.subplots(figsize=(10, 4))
        sorted_dist = dict(sorted(hw_dist.items(), key=lambda x: x[1], reverse=True)[:20])
        colors = ["red" if k == marked_bs else "steelblue" for k in sorted_dist]
        ax.bar(sorted_dist.keys(), sorted_dist.values(), color=colors)
        ax.axhline(
            y=random_p,
            color="gray",
            linestyle="--",
            label=f"Random baseline = {random_p:.4f}",
        )
        ax.set_xlabel("Measurement Outcome")
        ax.set_ylabel("Probability")
        ax.set_title(
            f"Best Hardware Run: {best_id} - Marked |{marked_bs}> (red) vs Others"
        )
        ax.legend()
        plt.xticks(rotation=45, fontsize=7)
        plt.tight_layout()

        out_path = os.path.join(OUTPUT_DIR, "search_verification.png")
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved {out_path}")
    else:
        print("No hardware distribution available for search verification.")
else:
    print("Hardware runs were disabled or no hardware results are available.")

---
## How to Run on IBM Hardware

| File | Content |
|------|---------|
| `results/grover/grover_results.csv` | Flat table with all metrics per run |
| `results/grover/grover_results.json` | Full distributions + metadata |
| `results/grover/success_prob_and_tvd.png` | Line plots of success prob & TVD |
| `results/grover/success_prob_bar.png` | Bar chart across all configs |
| `results/grover/search_verification.png` | Best-run histogram with marked state |
